# Azure AI Search Document Ingestion Pipeline

This notebook processes PDF documents and creates a searchable vector database in Azure AI Search, designed for RAG (Retrieval-Augmented Generation) scenarios where you need to search through document content using both keywords and semantic similarity.

## What This Notebook Does

### Main Purpose
Ingests PDF documents from Azure storage (OneLake/Fabric lakehouse), extracts their content using Azure Document Intelligence, chunks the text, generates embeddings, and indexes everything into Azure AI Search for hybrid search (keyword + vector search).

### Key Components

**1. Setup & Dependencies**
- Installs Azure SDK packages for Document Intelligence, Storage, Search, and OpenAI
- Configures placeholders for Azure Search, OpenAI, and Document Intelligence credentials

**2. Index Management**
- Creates an Azure AI Search index with fields for:
  - Text content and metadata (chunk_id, parent_id, title, URL, filepath)
  - Vector embeddings (1536 dimensions for text-embedding-ada-002)
  - Semantic search configuration
  - Vector search with HNSW algorithm

**3. Document Processing Pipeline**
- Reads PDFs from Azure storage using Spark
- Analyzes documents with Azure Document Intelligence (prebuilt-layout model) to extract text and structure
- Splits content into chunks using token-aware splitting (1024 tokens with 100-token overlap)
- Generates embeddings for each chunk using Azure OpenAI
- Uploads chunks to the search index with metadata

**4. File Tracking**
- Lists all files in the lakehouse directory
- Saves file metadata to a Delta table for tracking processed files

**5. Execution**
- Can process individual files or batch process entire directories

---

## Using This Notebook Outside of Microsoft Fabric

This notebook is designed for Microsoft Fabric but can be adapted for standalone use. Here are the required changes:

### 1. **Remove Spark Dependencies**
The notebook currently uses PySpark to read files from OneLake. For non-Fabric environments:

**Replace the Spark file reading code:**
```python
# FABRIC VERSION (current):
row = spark.read.format("binaryFile").load(file_URL).select("content").limit(1).head()
pdf_bytes = bytes(row[0])

# STANDALONE VERSION (use this instead):
from azure.storage.blob import BlobServiceClient

# For Azure Blob Storage:
blob_client = BlobServiceClient(
    account_url="https://<account>.blob.core.windows.net", 
    credential="<key>"
).get_blob_client(container="<container>", blob="<blob_path>")
pdf_bytes = blob_client.download_blob().readall()

# OR for local files:
with open(file_path, 'rb') as f:
    pdf_bytes = f.read()
```

### 2. **Update File Listing Logic**
Replace the `list_all_files()` function that uses `notebookutils`:

```python
# STANDALONE VERSION - For Azure Blob Storage:
from azure.storage.blob import ContainerClient

container_client = ContainerClient(
    account_url="https://<account>.blob.core.windows.net", 
    credential="<key>", 
    container_name="<container>"
)

all_files = []
for blob in container_client.list_blobs(name_starts_with="documents/"):
    all_files.append({
        "name": blob.name,
        "path": f"https://<account>.blob.core.windows.net/<container>/{blob.name}",
        "size": blob.size,
        "isFile": True
    })

# OR for local directory:
import os
all_files = []
for root, dirs, files in os.walk("./documents"):
    for file in files:
        if file.endswith('.pdf'):
            file_path = os.path.join(root, file)
            all_files.append({
                "name": file,
                "path": file_path,
                "size": os.path.getsize(file_path),
                "isFile": True
            })
```

### 3. **Remove Delta Table Tracking**
Replace the Delta table file tracking with a simple JSON or CSV file:

```python
# STANDALONE VERSION:
import json
from datetime import datetime

tracking_file = "processed_files.json"

# Load existing tracking
try:
    with open(tracking_file, 'r') as f:
        processed_files = json.load(f)
except FileNotFoundError:
    processed_files = {}

# Track processed file
processed_files[file_path] = {
    "processed_time": datetime.now().isoformat(),
    "size": file_size
}

# Save tracking
with open(tracking_file, 'w') as f:
    json.dump(processed_files, f, indent=2)
```

### 4. **Update File URL Format**
Change from OneLake ABFS URLs to standard URLs:
- **Fabric**: `abfss://guid@onelake.dfs.fabric.microsoft.com/...`
- **Azure Blob**: `https://account.blob.core.windows.net/container/path/file.pdf`
- **Local**: `./documents/file.pdf`

### 5. **Environment-Agnostic Code Example**
Here's a complete standalone `process_document()` function:

```python
def process_document_standalone(file_path: str, index_name: str, source_type: str = "local"):
    """
    Args:
        file_path: Path to PDF file
        index_name: Azure Search index name
        source_type: "local", "azure_blob", or "url"
    """
    # Read file based on source type
    if source_type == "local":
        with open(file_path, 'rb') as f:
            pdf_bytes = f.read()
    elif source_type == "azure_blob":
        from azure.storage.blob import BlobClient
        blob_client = BlobClient.from_blob_url(file_path, credential="<key>")
        pdf_bytes = blob_client.download_blob().readall()
    elif source_type == "url":
        import requests
        pdf_bytes = requests.get(file_path).content
    
    # Rest of processing remains the same
    client = DocumentIntelligenceClient(endpoint, AzureKeyCredential(key))
    poller = client.begin_analyze_document(
        "prebuilt-layout", pdf_bytes,
        content_type="application/pdf",
        features=["keyValuePairs"]
    )
    result = poller.result()
    process_afr_result(result, os.path.basename(file_path), index_name, file_path)
```

### 6. **Remove PySpark Imports**
Comment out or remove these lines from the imports cell:
```python
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import udf, col, current_timestamp
# from pyspark.sql.types import StringType, ArrayType, FloatType
```

### 7. **Remove Spark DataFrame Operations**
In the file tracking section, replace:
```python
# REMOVE:
spark_df = spark.createDataFrame(all_files)
spark_df = spark_df.withColumn("processed_time", current_timestamp())
spark_df.write.mode("append").format("delta").saveAsTable(table_name)

# REPLACE WITH:
import pandas as pd
df = pd.DataFrame(all_files)
df['processed_time'] = pd.Timestamp.now()
df.to_csv('processed_files.csv', mode='a', index=False, header=not os.path.exists('processed_files.csv'))
```

In [ ]:
%pip install azure-ai-documentintelligence 
%pip install azure-storage-blob 
%pip install azure-identity
%pip install azure-search
%pip install --upgrade azure-search-documents
%pip install tiktoken
%pip install langchain
%pip install openai

In [ ]:
import logging
from langchain.text_splitter import TextSplitter, MarkdownTextSplitter, RecursiveCharacterTextSplitter, PythonCodeTextSplitter
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
import os
import requests
import csv
import urllib.request, urllib.parse, urllib.error,  urllib
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, ArrayType, FloatType
import requests
import json
import datetime
from openai import AzureOpenAI

# Azure Search settings
SEARCH_ENDPOINT = "Your Azure Search Endpoint Here"
SEARCH_API_KEY = "Your Azure Search API Key Here"
SEARCH_INDEX = "Your Azure Search Index Name Here"
api_version = "?api-version=2025-05-01-preview"  # Using Preview version that supports vector search

# Azure OpenAI settings
OPEN_AI_ENDPOINT = "Your Azure OpenAI Endpoint Here"
OPEN_AI_KEY = "Your Azure OpenAI Key Here"
OPEN_AI_EMBEDDING_MODEL = "text-embedding-ada-002"
OPEN_AI_EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"  # This should be your actual deployment ID
OPEN_AI_API_VERSION = "2023-05-15"  # Using stable API version for embeddings

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ensure endpoint has trailing slash
if not SEARCH_ENDPOINT.endswith('/'):
    SEARCH_ENDPOINT += '/'

# Headers for Azure Search API calls
headers = {
    'Content-Type': 'application/json',
    'api-key': SEARCH_API_KEY
}

# Document Intelligence settings
endpoint = "Your Azure Document Intelligence Endpoint Here"
key = "Your Azure Document Intelligence Key Here"


SENTENCE_ENDINGS = [".", "!", "?"]
WORDS_BREAKS = list(reversed([",", ";", ":", " ", "(", ")", "[", "]", "{", "}", "\t", "\n"]))


StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 16, Finished, Available, Finished)

In [8]:
from typing import Callable, List, Dict, Optional, Generator, Tuple, Union
import tiktoken


class TokenEstimator(object):
    GPT2_TOKENIZER = tiktoken.get_encoding("gpt2")

    def estimate_tokens(self, text: Union[str, List]) -> int:

        return len(self.GPT2_TOKENIZER.encode(text, allowed_special="all"))

    def construct_tokens_with_size(self, tokens: str, numofTokens: int) -> str:
        newTokens = self.GPT2_TOKENIZER.decode(
            self.GPT2_TOKENIZER.encode(tokens, allowed_special="all")[:numofTokens]
        )
        return newTokens

TOKEN_ESTIMATOR = TokenEstimator()

StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 22, Finished, Available, Finished)

In [3]:
def delete_search_index(index_name):
    try:
        url = SEARCH_ENDPOINT + "indexes/" + index_name + api_version 
        response  = requests.delete(url, headers=headers)
        logging.info("Index deleted")
    except Exception as e:
        logging.info(e)

def create_search_index(index_schema,index_name):
    try:
        # Create Index
        url = SEARCH_ENDPOINT + "indexes/" + index_name + api_version
        response  = requests.put(url, headers=headers, json=index_schema)
        index = response.json()
        print(f"Index created {index_name} {index}")
    except Exception as e:
         print(f"error create_search_index {e}")

def add_document_to_index(page_idx, documents,search_index):
    try:
        #print(f"Adding documents to index: {documents}")
        print(f"Adding page_idx  {page_idx} - {len(documents['value'])} Documents added")
        url = SEARCH_ENDPOINT + "indexes/" + search_index + "/docs/index" + api_version
        print(url)
        response  = requests.post(url, headers=headers, json=documents)
        if response.status_code not in [200, 201]:
            logger.error(f"Error adding documents to index. Status code: {response.status_code}")
            logger.error(f"Error response: {response.text}")
        else:
            print(f"page_idx is {page_idx} - {len(documents['value'])} Documents added")
    except Exception as e:
        print(f"add_document_to_index {e}")

StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 17, Finished, Available, Finished)

In [4]:
import requests
import json

def create_the_index():
    # The full index schema JSON (as a multi-line string or Python dict)   
    index_schema_json = """
    {
      "name": "%s",
      "fields": [
        {
          "name": "chunk_id",
          "type": "Edm.String",
          "searchable": true,
          "filterable": true,
          "retrievable": true,
          "stored": true,
          "sortable": true,
          "facetable": true,
          "key": true,
          "analyzer": "keyword",
          "synonymMaps": []
        },
        {
          "name": "parent_id",
          "type": "Edm.String",
          "searchable": true,
          "filterable": true,
          "retrievable": true,
          "stored": true,
          "sortable": true,
          "facetable": true,
          "key": false,
          "synonymMaps": []
        },
        {
          "name": "content",
          "type": "Edm.String",
          "searchable": true,
          "filterable": false,
          "retrievable": true,
          "stored": true,
          "sortable": false,
          "facetable": false,
          "key": false,
          "synonymMaps": []
        },
        {
          "name": "title",
          "type": "Edm.String",
          "searchable": true,
          "filterable": true,
          "retrievable": true,
          "stored": true,
          "sortable": false,
          "facetable": false,
          "key": false,
          "synonymMaps": []
        },
        {
          "name": "url",
          "type": "Edm.String",
          "searchable": false,
          "filterable": false,
          "retrievable": true,
          "stored": true,
          "sortable": false,
          "facetable": false,
          "key": false,
          "synonymMaps": []
        },
        {
          "name": "filepath",
          "type": "Edm.String",
          "searchable": false,
          "filterable": false,
          "retrievable": true,
          "stored": true,
          "sortable": false,
          "facetable": false,
          "key": false,
          "synonymMaps": []
        },
        {
          "name": "contentVector",
          "type": "Collection(Edm.Single)",
          "searchable": true,
          "filterable": false,
          "retrievable": false,
          "stored": true,
          "sortable": false,
          "facetable": false,
          "key": false,
          "dimensions": 1536,
          "vectorSearchProfile": "default",
          "synonymMaps": []
        }
      ],
      "scoringProfiles": [],
      "suggesters": [],
      "analyzers": [],
      "normalizers": [],
      "tokenizers": [],
      "tokenFilters": [],
      "charFilters": [],
      "similarity": {
        "@odata.type": "#Microsoft.Azure.Search.BM25Similarity"
      },
      "semantic": {
        "configurations": [
          {
            "name": "default",
            "flightingOptIn": false,
            "rankingOrder": "BoostedRerankerScore",
            "prioritizedFields": {
              "titleField": { "fieldName": "title" },
              "prioritizedContentFields": [ { "fieldName": "content" } ],
              "prioritizedKeywordsFields": []
            }
          }
        ]
      },
      "vectorSearch": {
        "algorithms": [
          {
            "name": "default",
            "kind": "hnsw",
            "hnswParameters": {
              "metric": "cosine",
              "m": 4,
              "efConstruction": 400,
              "efSearch": 1000
            }
          }
        ],
        "profiles": [
          {
            "name": "default",
            "algorithm": "default",
            "vectorizer": "default"
          }
        ],
        "vectorizers": [
          {
            "name": "default",
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
              "resourceUri": "%s",
              "deploymentId": "%s",
              "apiKey": "%s",
              "modelName": "text-embedding-ada-002"
            }
          }
        ],
        "compressions": []
      }
    }    
    """ % (SEARCH_INDEX, OPEN_AI_ENDPOINT, OPEN_AI_EMBEDDING_DEPLOYMENT, OPEN_AI_KEY)

    try:
        # Convert JSON string to Python dict
        index_schema = json.loads(index_schema_json)
        logger.info("Successfully parsed index schema JSON")
        
        # Create Index
        headers = {
          "Content-Type": "application/json",
          "api-key": SEARCH_API_KEY
        }

        url = SEARCH_ENDPOINT + "indexes/" + SEARCH_INDEX + api_version
        logger.info(f"Creating index with URL: {url}")
        
        # Print the actual request payload for debugging
        logger.info("Request payload:")
        print(json.dumps(index_schema, indent=2))
        
        response = requests.put(url, headers=headers, json=index_schema)
        if response.status_code not in [200, 201]:
            logger.error(f"Error creating index. Status code: {response.status_code}")
            logger.error(f"Error response: {response.text}")
            # Try to parse and print the error response in a readable format
            try:
                error_json = response.json()
                if 'error' in error_json:
                    logger.error("Error details:")
                    logger.error(f"Code: {error_json['error'].get('code', 'Unknown')}")
                    logger.error(f"Message: {error_json['error'].get('message', 'No message')}")
                    if 'details' in error_json['error']:
                        logger.error("Additional details:")
                        for detail in error_json['error']['details']:
                            logger.error(f"- {detail.get('message', '')}")
            except Exception as parse_error:
                logger.error(f"Could not parse error response: {str(parse_error)}")
        else:
            index = response.json()
            logger.info(f"Index created successfully: {SEARCH_INDEX}")
            logger.info("Index details:")
            print(json.dumps(index, indent=2))
            
    except json.JSONDecodeError as je:
        logger.error(f"JSON parsing error: {str(je)}")
        logger.error("Invalid JSON schema:")
        print(index_schema_json)
    except Exception as e:
        logger.error(f"Error creating search index: {str(e)}")
        if hasattr(e, '__cause__') and e.__cause__:
            logger.error(f"Caused by: {str(e.__cause__)}")

StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 18, Finished, Available, Finished)

In [5]:
import uuid

def process_document(file_URL: str, index_name: str): 
    #read things from the request
    #api_version is not null
    
    # if len(api_version) == 0 :
    #    api_version = '?api-version=2021-04-30-Preview' 
    # else:
    #    logging.info('API version is: %s', api_version)
   
   
    SEARCH_INDEX = index_name    
    
    if(file_URL != ""):
      
      row = (spark.read.format("binaryFile").load(file_URL)
       .select("content").limit(1).head())
      pdf_bytes = bytes(row[0])

      print(f"file {file_URL} with {len(pdf_bytes)}")
      
            
      client = DocumentIntelligenceClient(endpoint, AzureKeyCredential(key))


      poller = client.begin_analyze_document(
          "prebuilt-layout",
          pdf_bytes,                           # your file bytes
          content_type="application/pdf",
          features=["keyValuePairs"]           # enable KVP extraction on layout in v4.0
          )
      result = poller.result()
      print(f"Processing result...this might take a few minutes...")      
      process_afr_result(result, "",index_name,file_URL) 
   
    print ("process completed")
   
        



def create_embeddings(text): 
    try:

        #print(f"creating embeddings for {text}")
        if text == "":
            return None
        
        client = AzureOpenAI(
        azure_endpoint = OPEN_AI_ENDPOINT, 
        api_key=OPEN_AI_KEY,  
        api_version="2024-02-15-preview"
        )

        return client.embeddings.create(input = [text], model=OPEN_AI_EMBEDDING_MODEL).data[0].embedding
    
    except Exception as e:
        print(f"Error create_embeddings: {e}")


def process_afr_result(result, filename,search_index,file_URL,content_chunk_overlap=100):
    print(f"Processing {filename } with {len(result.pages)} pages into Azure Search....this might take a few minutes depending on number of pages...")    
    for page_idx in range(len(result.pages)):
        docs = []
        #pageinfo = result.pages[page_idx]
        print(f"Processing page {page_idx} of {len(result.pages)}...")
        content_chunk = ""       
        
        for line_idx, line in enumerate(result.pages[page_idx].lines):            
            #print("...Line # {} has text content '{}'".format(line_idx,line.content.encode("utf-8")))
            # Assuming `line.content` is your text
            encoded_content = line.content.encode("utf-8")  # This will give you bytes
            decoded_content = encoded_content.decode("utf-8")  # This will give you string
            # Now you can add it to your content_chunk
            content_chunk += decoded_content + "\n"
             
        #now split the chunk        
        content_chunk_size=TOKEN_ESTIMATOR.estimate_tokens(content_chunk)
        content_chunk_size = 1024;
        if content_chunk_size > content_chunk_overlap:
           chunk_list = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                                separators=SENTENCE_ENDINGS + WORDS_BREAKS,
                                chunk_size=content_chunk_size, chunk_overlap=content_chunk_overlap).split_text(content_chunk)
        else:
            chunk_list = [content_chunk]
        
        parent_id = str(uuid.uuid4())        

        # Get the document title from Document Intelligence result
        # If available in metadata, use it; otherwise fallback to filename
        doc_title = None
        if hasattr(result, 'metadata') and result.metadata:
           doc_title = result.metadata.get('title', None)
            
        if not doc_title:
           # Fallback to filename without extension
           file_name = os.path.splitext(os.path.basename(file_URL))[0]
           doc_title = file_name

        print(f"Processing Parent Document  {parent_id}...")
        chunk_idx = 1
        for chunked_content in chunk_list:
            chunk_id = f"{parent_id}-{chunk_idx}"  # Make sure chunk_id is unique
            chunk_size = TOKEN_ESTIMATOR.estimate_tokens(chunked_content)
            #print(f"content {chunked_content} with size {chunk_size}")            
            embeddings = create_embeddings(chunked_content)                         
            parent_id = str(uuid.uuid4())
            search_doc = {
                    "parent_id": parent_id,
                    "chunk_id":  chunk_id,
                    "content": content_chunk,
                    "url" : file_URL,
                    "title":  doc_title,
                    "filepath": file_URL.split('/')[-1],                    
                    "contentVector": embeddings
            }
            docs.append(search_doc)            
            chunk_idx += 1  
       
        print(f"Processed {len(docs)} documents for page {page_idx}.")
        add_document_to_index(page_idx, {"value": docs},search_index)
        #create_chunked_data_files(page_idx, search_doc)

def create_chunked_data_files(page_idx, search_doc):
    try:
        output_path = os.path.join(os.getcwd(), "data-files", f'{page_idx}-data.csv')
        with open(output_path, 'w') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([search_doc['id'], search_doc['text']])
            
    except Exception as e:
        logging.info(e)



StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 19, Finished, Available, Finished)

In [6]:

#delete and recreate the index
delete_search_index(SEARCH_INDEX)
create_the_index()

StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 20, Finished, Available, Finished)

{
  "name": "shortcuts",
  "fields": [
    {
      "name": "chunk_id",
      "type": "Edm.String",
      "searchable": true,
      "filterable": true,
      "retrievable": true,
      "stored": true,
      "sortable": true,
      "facetable": true,
      "key": true,
      "analyzer": "keyword",
      "synonymMaps": []
    },
    {
      "name": "parent_id",
      "type": "Edm.String",
      "searchable": true,
      "filterable": true,
      "retrievable": true,
      "stored": true,
      "sortable": true,
      "facetable": true,
      "key": false,
      "synonymMaps": []
    },
    {
      "name": "content",
      "type": "Edm.String",
      "searchable": true,
      "filterable": false,
      "retrievable": true,
      "stored": true,
      "sortable": false,
      "facetable": false,
      "key": false,
      "synonymMaps": []
    },
    {
      "name": "title",
      "type": "Edm.String",
      "searchable": true,
      "filterable": true,
      "retrievable": true,
      "stor

In [ ]:
FileURL = ("abfss://974a6c8a-851e-4255-ac4b-8fc77b981dd2@onelake.dfs.fabric.microsoft.com/02d40078-6876-40fe-a768-2a3e19d5db03/Files/codeserendipity/20020042318.pdf")
result = process_document(FileURL,SEARCH_INDEX)

## Read and process the files

In [10]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp

#dir_path = '/lakehouse/Sales_and_stuff_lakehouse/Files/documents/documents'

dir_path = "/lakehouse/Sales_and_stuff_lakehouse/Files/codeserendipity"
dir_path = "Files/codeserendipity"


def list_all_files(dir_path, utils=None):
    """
    Recursively list all files under dir_path.
    Returns a list of dicts with name, path, size, and isDir/isFile flags.
    """
    # Prefer new API name; fall back to mssparkutils if needed
    if utils is None:
        try:
            _utils = notebookutils
        except NameError:
            from notebookutils import mssparkutils as _utils
        utils = _utils

    out = []
    stack = [dir_path]
    while stack:
        current = stack.pop()
        for entry in utils.fs.ls(current):            
            info = {
                "name": entry.name,
                "path": entry.path,
                "size": entry.size,
                "isDir": entry.isDir,
                "isFile": entry.isFile,                
                #"modified_time": datetime.fromtimestamp(entry.modifyTime / 1000)               
            }
            out.append(info)
            if entry.isDir:
                stack.append(entry.path)
    return out

all_files = list_all_files(dir_path)


StatementMeta(, cd169896-8638-4b97-a8e2-092a83ff5b42, 24, Finished, Available, Finished)

In [29]:
from pyspark.sql.utils import AnalysisException
#########################################################
#### Save the files to a table

table_exists = True
table_name = "sales_and_stuff_lakehouse.ai_search_processed_files"
spark_df = spark.createDataFrame(all_files)
spark_df = spark_df.withColumn("processed_time", current_timestamp())
try:
    # Try reading the table
    existing_df = spark.table(table_name)
    print("table exists")

    # Append new data   
    spark_df.write.mode("append").format("delta").saveAsTable(table_name)

except AnalysisException:
    # Table doesn't exist, so create it
    table_exists = False
    print("table does not")   
    spark_df.write.format("delta").saveAsTable(table_name)



StatementMeta(, dfb27ba4-391a-47ee-8af6-62463be3884e, 73, Finished, Available, Finished)

table exists


In [ ]:
#########################################################
### Process the files
#### To-Do:  process only files that are new or that have been modified since last processed

data_files = [f for f in all_files]
len(data_files), data_files[:5]
for item in data_files:
    FileURL = (item['path'])
    result = process_document(FileURL,SEARCH_INDEX)
    